# Chapter 12: Type

> *"Well-typed programs cannot go wrong."*  
> — **Robin Milner**, *A Theory of Type Polymorphism in Programming*
> (1978)

------------------------------------------------------------------------

In the physical silicon of the machine, there is no such thing as an
integer, a floating-point number, a string, or a lambda. There is only a
sea of binary voltages: ones and zeroes arranged in groups of 8, 16, 32,
or 64 bits.

Consider the 32-bit hex word `0x40490FDB`.

- If the CPU passes this word to the floating-point unit (FPU), it is
  decoded under the IEEE-754 standard as the real number
  $\approx 3.14159265$.
- If the CPU passes it to an integer arithmetic instruction, it
  evaluates to the signed integer $1,078,530,011$.
- If sent to an ASCII terminal buffer, it renders as the four-character
  text sequence `@I\x0f\xdb`.
- If loaded into the Program Counter register, it is executed as a
  branch instruction.

The bit pattern itself contains zero intrinsic meaning. **Meaning is
imposed upon bits by the Type System.**

In Part I of this book, we experienced Python's dynamic and nominal type
system. In this chapter, we enter the domain of **Programming Language
Theory (PLT)** and explore the static, nominal, compile-time type system
of Modern C++.

We will explore what types are from a mathematical perspective, master
the algebra of **Product Types** and **Sum Types**, leverage **C++20
Concepts** to express formal semantic constraints, and implement a
complete **Symbolic Algebraic Engine and Type Inspector** that performs
symbolic differentiation, semantic validation, and constant reduction
using pattern-matched tagged unions.

### 12.1 The Ontological Question: What is a Type?

From the perspective of Programming Language Theory, a **Type** is not
merely a label or a byte size. A type is a formal mathematical predicate
that establishes three things:

1.  **A Set of Permissible Values:** The domain of valid states the data
    may inhabit (e.g., `bool` $\in \{\text{true}, \text{false}\}$,
    `uint8_t` $\in [0, 255]$).
2.  **A Set of Permissible Operations:** The legal transformations
    allowed upon those values (e.g., you can divide two `double` values,
    but you cannot compute the bitwise XOR of two strings).
3.  **A Memory Representation:** The exact spatial layout, alignment,
    and interpretation rule mapping the type's values to contiguous
    physical bytes in RAM.

#### 12.1.1 The Typing Spectrum

Programming languages enforce types along two major orthogonal axes (see
Figure 12–1): **Static vs Dynamic** and **Nominal vs Structural**.

<figure id="fig:typing-spectrum">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIwIDAgOTAwIDQyMCIgd2lkdGg9IjEwMCUiIGhlaWdodD0iYXV0byIgeG1s
bnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KICA8ZGVmcz4KICAgIDxzdHlsZT4KICAg
ICAgOnJvb3QgewogICAgICAgIC0tc3ZnLWJnOiB2YXIoLS1iZywgI2ZmZmZmZik7CiAgICAgICAg
LS1zdmctZmc6IHZhcigtLWZnLCAjMDAwMDAwKTsKICAgICAgICAtLXN2Zy10aW50LTEtZmFsbGJh
Y2s6ICNjY2NjY2M7IC8qIDIwJSBibGFjayArIDgwJSB3aGl0ZSAqLwogICAgICAgIC0tc3ZnLXRp
bnQtMi1mYWxsYmFjazogIzk5OTk5OTsgLyogNDAlIGJsYWNrICsgNjAlIHdoaXRlICovCiAgICAg
ICAgLS1zdmctdGludC0xOiBjb2xvci1taXgoaW4gc3JnYiwgdmFyKC0tc3ZnLWZnKSAyMCUsIHZh
cigtLXN2Zy1iZykgODAlKTsKICAgICAgICAtLXN2Zy10aW50LTI6IGNvbG9yLW1peChpbiBzcmdi
LCB2YXIoLS1zdmctZmcpIDQwJSwgdmFyKC0tc3ZnLWJnKSA2MCUpOwogICAgICB9CiAgICAgIHRl
eHQgICB7IGZvbnQtZmFtaWx5OiAiQXJpYWwgTmFycm93IiwgRnJlZVNhbnMsIHNhbnMtc2VyaWY7
IH0KICAgIDwvc3R5bGU+CiAgICA8ZmlsdGVyIGlkPSJ0eXBlLXNoYWRvdyIgeD0iLTUlIiB5PSIt
NSUiIHdpZHRoPSIxMTAlIiBoZWlnaHQ9IjExMCUiPgogICAgICA8ZmVEcm9wU2hhZG93IGR4PSIy
IiBkeT0iMiIgc3RkRGV2aWF0aW9uPSIzIiBmbG9vZC1vcGFjaXR5PSIwLjE1Ii8+CiAgICA8L2Zp
bHRlcj4KICA8L2RlZnM+CiAgPCEtLSBRdWFkcmFudCBHcmlkIExpbmVzIC0tPgogIDxsaW5lIHgx
PSI0NTAiIHkxPSI0MCIgeDI9IjQ1MCIgeTI9IjM4MCIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBz
dHJva2Utd2lkdGg9IjIiIHN0cm9rZS1kYXNoYXJyYXk9IjQsNCIvPgogIDxsaW5lIHgxPSI0MCIg
eTE9IjIxMCIgeDI9Ijg4MCIgeTI9IjIxMCIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Ut
d2lkdGg9IjIiIHN0cm9rZS1kYXNoYXJyYXk9IjQsNCIvPgogIDwhLS0gQXhpcyBMYWJlbHMgLS0+
CiAgPHRleHQgeD0iNDUwIiB5PSIzMCIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQi
IGZpbGw9InZhcigtLXN2Zy1mZykiIHRleHQtYW5jaG9yPSJtaWRkbGUiPlNUQVRJQyBUWVBJTkcg
KENoZWNrZWQgYXQgQ29tcGlsZSBUaW1lKTwvdGV4dD4KICA8dGV4dCB4PSI0NTAiIHk9IjQwNSIg
Zm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiIHRl
eHQtYW5jaG9yPSJtaWRkbGUiPkRZTkFNSUMgVFlQSU5HIChDaGVja2VkIGF0IFJ1bnRpbWUpPC90
ZXh0PgogIDx0ZXh0IHg9IjMwIiB5PSIyMTUiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJi
b2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB0cmFuc2Zvcm09
InJvdGF0ZSgtOTAgMzAgMjE1KSI+Tk9NSU5BTDwvdGV4dD4KICA8dGV4dCB4PSI4NzAiIHk9IjIw
MCIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZyki
IHRleHQtYW5jaG9yPSJtaWRkbGUiIHRyYW5zZm9ybT0icm90YXRlKDkwIDg3MCAyMTUpIj5TVFJV
Q1RVUkFMPC90ZXh0PgogIDwhLS0gVG9wLUxlZnQ6IFN0YXRpYyArIE5vbWluYWwgKE1vZGVybiBD
KyssIFJ1c3QsIEphdmEpIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDYwLCA1NSkiIGZp
bHRlcj0idXJsKCN0eXBlLXNoYWRvdykiPgogICAgPHJlY3Qgd2lkdGg9IjM2MCIgaGVpZ2h0PSIx
MzUiIHJ4PSI4IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxi
YWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiLz4KICAgIDx0
ZXh0IHg9IjE1IiB5PSIyOCIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9
InZhcigtLXN2Zy1mZykiPlN0YXRpYyAmYW1wOyBOb21pbmFsIChNb2Rlcm4gQysrKTwvdGV4dD4K
ICAgIDx0ZXh0IHg9IjE1IiB5PSI1MiIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZn
KSI+4oCiIFR5cGVzIHZlcmlmaWVkIGF0IGNvbXBpbGUtdGltZTwvdGV4dD4KICAgIDx0ZXh0IHg9
IjE1IiB5PSI3MCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+4oCiIEVxdWl2
YWxlbmNlIGRlZmluZWQgYnkgZXhwbGljaXQgbmFtZS90YWc8L3RleHQ+CiAgICA8dGV4dCB4PSIx
NSIgeT0iODgiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPuKAoiBaZXJvIHJ1
bnRpbWUgb3ZlcmhlYWQgKFplcm8tQ29zdCk8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iMTA2
IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+
c3RydWN0IE1ldGVycyB7IGRvdWJsZSB2OyB9OzwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSIx
MjIiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcp
Ij5zdHJ1Y3QgU2Vjb25kcyB7IGRvdWJsZSB2OyB9OyAvLyBJbmNvbXBhdGlibGUhPC90ZXh0Pgog
IDwvZz4KICA8IS0tIFRvcC1SaWdodDogU3RhdGljICsgU3RydWN0dXJhbCAoVHlwZVNjcmlwdCwg
R28gaW50ZXJmYWNlcykgLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoNDgwLCA1NSkiIGZp
bHRlcj0idXJsKCN0eXBlLXNoYWRvdykiPgogICAgPHJlY3Qgd2lkdGg9IjM4MCIgaGVpZ2h0PSIx
MzUiIHJ4PSI4IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxi
YWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiLz4KICAgIDx0
ZXh0IHg9IjE1IiB5PSIyOCIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9
InZhcigtLXN2Zy1mZykiPlN0YXRpYyAmYW1wOyBTdHJ1Y3R1cmFsIChUeXBlU2NyaXB0KTwvdGV4
dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSI1MiIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3Zn
LWZnKSI+4oCiIFR5cGVzIHZlcmlmaWVkIGF0IGNvbXBpbGUtdGltZTwvdGV4dD4KICAgIDx0ZXh0
IHg9IjE1IiB5PSI3MCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+4oCiIEVx
dWl2YWxlbmNlIGRlZmluZWQgYnkgZmllbGQgc2hhcGU8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIg
eT0iODgiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPuKAoiBFcmFzZWQgY29t
cGxldGVseSBpbiBKYXZhU2NyaXB0IG91dHB1dDwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSIx
MDYiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcp
Ij50eXBlIFBvaW50ID0geyB4OiBudW1iZXIsIHk6IG51bWJlciB9OzwvdGV4dD4KICAgIDx0ZXh0
IHg9IjE1IiB5PSIxMjIiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2
YXIoLS1zdmctZmcpIj50eXBlIFZlYzIgID0geyB4OiBudW1iZXIsIHk6IG51bWJlciB9OyAvLyBJ
ZGVudGljYWwhPC90ZXh0PgogIDwvZz4KICA8IS0tIEJvdHRvbS1MZWZ0OiBEeW5hbWljICsgTm9t
aW5hbCAoUHl0aG9uLCBSdWJ5LCBTbWFsbHRhbGspIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNs
YXRlKDYwLCAyMzApIiBmaWx0ZXI9InVybCgjdHlwZS1zaGFkb3cpIj4KICAgIDxyZWN0IHdpZHRo
PSIzNjAiIGhlaWdodD0iMTM1IiByeD0iOCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0t
c3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0
aD0iMS41Ii8+CiAgICA8dGV4dCB4PSIxNSIgeT0iMjgiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2Vp
Z2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5EeW5hbWljICZhbXA7IE5vbWluYWwgKFB5
dGhvbik8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iNTIiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9
InZhcigtLXN2Zy1mZykiPuKAoiBUeXBlcyBjaGVja2VkIGR1cmluZyBwcm9ncmFtIGV4ZWN1dGlv
bjwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSI3MCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFy
KC0tc3ZnLWZnKSI+4oCiIFZhcmlhYmxlcyBjYXJyeSBydW50aW1lIGNsYXNzIG1ldGFkYXRhIHRh
Z3M8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iODgiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZh
cigtLXN2Zy1mZykiPuKAoiBpc2luc3RhbmNlKG9iaiwgQ2xhc3NOYW1lKSBjaGVja3M8L3RleHQ+
CiAgICA8dGV4dCB4PSIxNSIgeT0iMTEyIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9s
ZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+VHlwZUVycm9yIHJhaXNlZCBpZiBtZXRob2QgbWlzc2lu
ZzwvdGV4dD4KICA8L2c+CiAgPCEtLSBCb3R0b20tUmlnaHQ6IER5bmFtaWMgKyBTdHJ1Y3R1cmFs
IChKYXZhU2NyaXB0LCBEdWNrIFR5cGluZykgLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUo
NDgwLCAyMzApIiBmaWx0ZXI9InVybCgjdHlwZS1zaGFkb3cpIj4KICAgIDxyZWN0IHdpZHRoPSIz
ODAiIGhlaWdodD0iMTM1IiByeD0iOCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3Zn
LXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0i
MS41Ii8+CiAgICA8dGV4dCB4PSIxNSIgeT0iMjgiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0
PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5EeW5hbWljICZhbXA7IFN0cnVjdHVyYWwgKEph
dmFTY3JpcHQgLyBEdWNrKTwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSI1MiIgZm9udC1zaXpl
PSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+4oCiICJJZiBpdCB3YWxrcyBhbmQgcXVhY2tzIGxp
a2UgYSBkdWNrLi4uIjwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSI3MCIgZm9udC1zaXplPSIx
NCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+4oCiIFByb3BlcnR5IGxvb2t1cCBieSBzdHJpbmcga2V5
IGF0IHJ1bnRpbWU8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iODgiIGZvbnQtc2l6ZT0iMTQi
IGZpbGw9InZhcigtLXN2Zy1mZykiPuKAoiBIaWdoZXN0IGZsZXhpYmlsaXR5LCBoaWdoZXN0IHJ1
bnRpbWUgZmFpbHVyZSByaXNrPC90ZXh0PgogICAgPHRleHQgeD0iMTUiIHk9IjExMiIgZm9udC1z
aXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPm9iai5mbHko
KSAtPiBVbmNhdWdodCBUeXBlRXJyb3I6IHVuZGVmaW5lZDwvdGV4dD4KICA8L2c+Cjwvc3ZnPg==
" alt="Figure 12–1: Orthogonal Type Classifications" />
<figcaption>Figure 12–1: Orthogonal Type Classifications</figcaption>
</figure>

#### 12.1.2 Type Safety and the Curry-Howard Isomorphism

In PLT, the **Curry-Howard Isomorphism** establishes a profound
equivalence between mathematics and computer science:

- A **Type** corresponds to a mathematical **Proposition** (a Theorem).
- A **Program / Expression** corresponds to a **Proof** of that
  Proposition.
- A **Type Checker** is an automated **Proof Verifier**.

When a C++ program compiles successfully without type errors, the
compiler has mathematically verified that the operations performed on
every value are valid within the rules of the system. In Robin Milner's
words: *a well-typed program cannot get stuck*.

### 12.2 Algebraic Data Types (ADTs)

In type theory, types can be combined algebraically through operations
analogous to multiplication (**Product Types**) and addition (**Sum
Types**). The *Cardinality* of a type, denoted $|T|$, is the total
number of distinct possible values that inhabit the type.

#### 12.2.1 Product Types (Logical Conjunction: "AND")

A **Product Type** compounds multiple types together. An instance of a
product type contains a value of type $A$ **AND** a value of type $B$.

The cardinality of a product type is the product of the cardinalities of
its constituents:

<span id="eq:eq12-1">$$|A \times B| = |A| \times |B|\qquad{(12–1)}$$</span>

In C++, `struct`, `class`, `std::pair`, and `std::tuple` are Product
Types.

``` cpp
struct RGBColour {
    std::uint8_t r; // |uint8_t| = 256
    std::uint8_t g; // |uint8_t| = 256
    std::uint8_t b; // |uint8_t| = 256
};
// Cardinality: |RGBColour| = 256 * 256 * 256 = 16,777,216 distinct states
```

#### 12.2.2 Sum Types (Logical Disjunction: "OR")

A **Sum Type** (also known as a *Tagged Union*, *Coproduct*, or
*Variant*) represents an exclusive choice. An instance of a sum type
holds a value of type $A$ **OR** a value of type $B$, but never both
simultaneously.

The cardinality of a sum type is the sum of the cardinalities of its
constituents:

<span id="eq:eq12-2">$$|A + B| = |A| + |B|\qquad{(12–2)}$$</span>

In Modern C++, Sum Types are represented by `std::variant<Ts...>`,
`std::optional<T>` ($|T| + 1$), and `std::expected<T, E>`.

``` cpp
#include <variant>
#include <string>

// A Sum Type representing network connection status
using ConnectionStatus = std::variant<
    bool,        // Disconnected (2 states: false) / Connected (true)
    int,         // Error Code (2^32 states)
    std::string  // Reconnection Hostname (Infinite states)
>;
```

#### 12.2.3 The Memory Anatomy: Product vs Sum Type

Understanding the physical memory layout of Product versus Sum types
highlights why Sum Types are memory-efficient (as compared in
Figure 12–2):

<figure id="fig:sum-vs-product">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIwIDAgODUwIDMyMCIgd2lkdGg9IjEwMCUiIGhlaWdodD0iYXV0byIgeG1s
bnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KICA8ZGVmcz4KICAgIDxzdHlsZT4KICAg
ICAgOnJvb3QgewogICAgICAgIC0tc3ZnLWJnOiB2YXIoLS1iZywgI2ZmZmZmZik7CiAgICAgICAg
LS1zdmctZmc6IHZhcigtLWZnLCAjMDAwMDAwKTsKICAgICAgICAtLXN2Zy10aW50LTEtZmFsbGJh
Y2s6ICNjY2NjY2M7IC8qIDIwJSBibGFjayArIDgwJSB3aGl0ZSAqLwogICAgICAgIC0tc3ZnLXRp
bnQtMi1mYWxsYmFjazogIzk5OTk5OTsgLyogNDAlIGJsYWNrICsgNjAlIHdoaXRlICovCiAgICAg
ICAgLS1zdmctdGludC0xOiBjb2xvci1taXgoaW4gc3JnYiwgdmFyKC0tc3ZnLWZnKSAyMCUsIHZh
cigtLXN2Zy1iZykgODAlKTsKICAgICAgICAtLXN2Zy10aW50LTI6IGNvbG9yLW1peChpbiBzcmdi
LCB2YXIoLS1zdmctZmcpIDQwJSwgdmFyKC0tc3ZnLWJnKSA2MCUpOwogICAgICB9CiAgICAgIHRl
eHQgICB7IGZvbnQtZmFtaWx5OiAiQXJpYWwgTmFycm93IiwgRnJlZVNhbnMsIHNhbnMtc2VyaWY7
IH0KICAgIDwvc3R5bGU+CiAgICA8ZmlsdGVyIGlkPSJtZW0tYWR0LXNoYWRvdyIgeD0iLTUlIiB5
PSItNSUiIHdpZHRoPSIxMTAlIiBoZWlnaHQ9IjExMCUiPgogICAgICA8ZmVEcm9wU2hhZG93IGR4
PSIyIiBkeT0iMiIgc3RkRGV2aWF0aW9uPSIzIiBmbG9vZC1vcGFjaXR5PSIwLjEyIi8+CiAgICA8
L2ZpbHRlcj4KICA8L2RlZnM+CiAgPCEtLSAxLiBQcm9kdWN0IFR5cGUgTGF5b3V0IChTdHJ1Y3Qp
IC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDQwLCA0MCkiIGZpbHRlcj0idXJsKCNtZW0t
YWR0LXNoYWRvdykiPgogICAgPHJlY3Qgd2lkdGg9IjM4MCIgaGVpZ2h0PSIyNDAiIHJ4PSI4IiBm
aWxsPSJ2YXIoLS1zdmctdGludC0yLCB2YXIoLS1zdmctdGludC0yLWZhbGxiYWNrKSkiIHN0cm9r
ZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiLz4KICAgIDx0ZXh0IHg9IjE1IiB5
PSIyNCIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1m
ZykiPlBST0RVQ1QgVFlQRTogc3RydWN0IFBheWxvYWQ8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIg
eT0iNTUiIGZvbnQtc2l6ZT0iMTIiIGZpbGw9InZhcigtLXN2Zy1mZykiPlNpemUgPSBzaXplb2Yo
aW50KSArIHNpemVvZihkb3VibGUpICsgc2l6ZW9mKGNoYXJbMTZdKSA9IDMyQjwvdGV4dD4KICAg
IDwhLS0gRmllbGQgMSAtLT4KICAgIDxyZWN0IHg9IjE1IiB5PSI3MCIgd2lkdGg9IjM1MCIgaGVp
Z2h0PSI0MCIgcng9IjQiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEt
ZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgICA8dGV4dCB4PSIyNSIgeT0i
OTUiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPmludCBpZDsgKDQgQnl0ZXMp
IFtPZmZzZXQ6IDBdPC90ZXh0PgogICAgPCEtLSBGaWVsZCAyIC0tPgogICAgPHJlY3QgeD0iMTUi
IHk9IjExNSIgd2lkdGg9IjM1MCIgaGVpZ2h0PSI0MCIgcng9IjQiIGZpbGw9InZhcigtLXN2Zy10
aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcp
Ii8+CiAgICA8dGV4dCB4PSIyNSIgeT0iMTQwIiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1z
dmctZmcpIj5kb3VibGUgdmFsdWU7ICg4IEJ5dGVzKSBbT2Zmc2V0OiA4XTwvdGV4dD4KICAgIDwh
LS0gRmllbGQgMyAtLT4KICAgIDxyZWN0IHg9IjE1IiB5PSIxNjAiIHdpZHRoPSIzNTAiIGhlaWdo
dD0iNTAiIHJ4PSI0IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZh
bGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogICAgPHRleHQgeD0iMjUiIHk9IjE5
MCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+Y2hhciBsYWJlbFsxNl07ICgx
NiBCeXRlcykgW09mZnNldDogMTZdPC90ZXh0PgogIDwvZz4KICA8IS0tIDIuIFN1bSBUeXBlIExh
eW91dCAoc3RkOjp2YXJpYW50KSAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSg0NTAsIDQw
KSIgZmlsdGVyPSJ1cmwoI21lbS1hZHQtc2hhZG93KSI+CiAgICA8cmVjdCB3aWR0aD0iNDAwIiBo
ZWlnaHQ9IjI0MCIgcng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTIsIHZhcigtLXN2Zy10aW50
LTItZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIv
PgogICAgPHRleHQgeD0iMTUiIHk9IjI0IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9s
ZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+U1VNIFRZUEU6IHN0ZDo6dmFyaWFudCZsdDtpbnQsIGRv
dWJsZSwgY2hhclsxNl0mZ3Q7PC90ZXh0PgogICAgPHRleHQgeD0iMTUiIHk9IjU1IiBmb250LXNp
emU9IjEyIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5TaXplID0gbWF4KHNpemVvZihUcy4uLikpICsg
c2l6ZW9mKHR5cGVfaW5kZXgpID0gMjRCPC90ZXh0PgogICAgPCEtLSBEaXNjcmltaW5hbnQgVGFn
IC0tPgogICAgPHJlY3QgeD0iMTUiIHk9IjcwIiB3aWR0aD0iMzcwIiBoZWlnaHQ9IjQwIiByeD0i
NCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBz
dHJva2U9InZhcigtLXN2Zy1mZykiLz4KICAgIDx0ZXh0IHg9IjI1IiB5PSI5NSIgZm9udC1zaXpl
PSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPkRpc2NyaW1pbmF0
b3IgLyBUeXBlIEluZGV4IChzaXplX3QpIFtPZmZzZXQ6IDBdPC90ZXh0PgogICAgPCEtLSBTaGFy
ZWQgU3RvcmFnZSBCdWZmZXIgLS0+CiAgICA8cmVjdCB4PSIxNSIgeT0iMTE1IiB3aWR0aD0iMzcw
IiBoZWlnaHQ9Ijk1IiByeD0iNCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRp
bnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICAgIDx0ZXh0IHg9IjI1
IiB5PSIxNDAiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1z
dmctZmcpIj5TaGFyZWQgTWVtb3J5IEJ1ZmZlciAoMTYgQnl0ZXMpPC90ZXh0PgogICAgPHRleHQg
eD0iMjUiIHk9IjE2NSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+T3Zlcmxh
eTogaW50ICg0QikgT1IgZG91YmxlICg4QikgT1IgY2hhclsxNl0gKDE2Qik8L3RleHQ+CiAgICA8
dGV4dCB4PSIyNSIgeT0iMTg1IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj4o
T25seSBPTkUgYWN0aXZlIHR5cGUgYXQgYW55IGluc3RhbnQpPC90ZXh0PgogIDwvZz4KPC9zdmc+
" alt="Figure 12–2: Product and Sum Types Compared" />
<figcaption>Figure 12–2: Product and Sum Types Compared</figcaption>
</figure>

### 12.3 Pattern Matching with `std::visit` and the Overloaded Idiom

In languages like Rust, OCaml, or Haskell, sum types are unwrapped using
built-in pattern matching (`match val { ... }`). In Modern C++, we
achieve type-safe, exhaustive compile-time pattern matching using
`std::visit` paired with the standard **Overloaded struct idiom**.

#### 12.3.1 Constructing the `overloaded` Pattern Matcher

Using C++17 variadic templates, class template argument deduction
(CTAD), and parameter pack expansion of `using` declarations, we
construct an overload pattern matcher in just two lines of code:

``` cpp
template <class... Ts>
struct overloaded : Ts... {
    using Ts::operator()...;
};

// Explicit deduction guide (optional in C++20, required in C++17)
template <class... Ts>
overloaded(Ts...) -> overloaded<Ts...>;
```

When instantiated with multiple lambdas, `overloaded` inherits the
`operator()` call operators of each lambda, synthesising a single
callable object with overloaded dispatch resolution handled entirely at
compile time!

``` cpp
#include <variant>
#include <string>
#include <iostream>
#include <format>

using Primitive = std::variant<int, double, std::string>;

void inspect(const Primitive& p) {
    std::visit(overloaded{
        [](int val) { 
            std::cout << std::format("Integer literal: {}\n", val); 
        },
        [](double val) { 
            std::cout << std::format("Real floating-point: {:.4f}\n", val); 
        },
        [](const std::string& val) { 
            std::cout << std::format("Text string: \"{}\"\n", val); 
        }
    }, p);
}
```

*Exhaustiveness Guarantee:* If you add a new type to the `std::variant`
and forget to handle it inside `std::visit`, the compiler will fail to
compile your program, preventing unhandled edge cases.

### 12.4 Modern C++ Concepts: Formal Compile-Time Contracts

Prior to C++20, template metaprogramming relied on **SFINAE**
(*Substitution Failure Is Not An Error*) and `std::enable_if_t`. When a
type failed a template requirement, the compiler emitted hundreds of
lines of cryptic, undecipherable error messages.

**C++20 Concepts** transform templates into formally constrained,
verifiable semantic contracts.

``` cpp
#include <concepts>
#include <type_traits>

// Define a semantic concept for Numeric types
template <typename T>
concept Numeric = std::is_arithmetic_v<T> && !std::is_same_v<T, bool>;

// Define a concept for types supporting algebraic field addition
template <typename T>
concept Addable = requires(T a, T b) {
    { a + b } -> std::convertible_to<T>;
};

// Constrained generic function
template <Numeric T>
[[nodiscard]] constexpr T square(T val) noexcept {
    return val * val;
}
```

If a developer attempts to call `square("invalid")`, the compiler
immediately outputs an explicit, actionable error:

```
error: constraints not satisfied for class template 'square'
note: because 'const char*' does not satisfy 'Numeric'
note: because 'std::is_arithmetic_v<const char*>' evaluated to false
```

### 12.5 Experimental Implementation in Modern C++: The Symbolic Algebraic Type Engine

To demonstrate the power of Modern C++ type systems, we will construct a
**Symbolic Algebraic Engine**.

Our system will:

1.  Model complex mathematical expressions as an AST using **Sum Types**
    (`std::variant`).
2.  Implement **Type-Driven Semantic Inspection** to enforce semantic
    correctness.
3.  Perform **Symbolic Constant Folding and Reduction** (e.g.,
    $x + 0 \to x$, $x \times 0 \to 0$, $x \times 1 \to x$).
4.  Compute exact **Analytical Symbolic Derivatives**
    ($\frac{d}{dx} f(x)$) using structural calculus rules (Product Rule,
    Chain Rule, Power Rule).

- Listing 12–1 implements the overall defining architecture for the
  expressions type-system.
- Listing 12–2 uses `std::visit` to pretty-print mathematical
  expressions as strings and evaluate them.
- Listing 12–3 utilizes pattern matching over variants to perform
  algebraic reductions: $$\begin{aligned}
    x + 0 &\to x \\
    x \times 0 &\to 0 \\
    x \times 1 &\to x \\
    x^1 &\to x \\
    x^0 &\to 1
    \end{aligned}$$
- Listing 12–4 obeys the formal structural rules of differential
  calculus:
  - Constant Rule: $\frac{d}{dx}[c] = 0$
  - Variable Rule: $\frac{d}{dx}[x] = 1$, $\frac{d}{dx}[y] = 0$
  - Sum Rule: $\frac{d}{dx}[u + v] = u' + v'$
  - Product Rule: $\frac{d}{dx}[u \cdot v] = u' v + u v'$
  - Quotient Rule:
    $\frac{d}{dx}\left[\frac{u}{v}\right] = \frac{u' v - u v'}{v^2}$
  - Power Rule: $\frac{d}{dx}[u^n] = n \cdot u^{n-1} \cdot u'$
- Listing 12–5 creates a demonstration and verification harness for this
  code.

Listing 12–1: Expression Types Architecture (SymbolicEngine.hpp):

In [ ]:
#ifndef SYMBOLIC_ENGINE_HPP
#define SYMBOLIC_ENGINE_HPP

#include <memory>
#include <string>
#include <variant>
#include <iostream>
#include <format>
#include <cmath>
#include <stdexcept>
#include <concepts>

namespace sym {

// Forward declaration of recursive AST Node
struct Expression;

// Pattern Matcher Helper
template <class... Ts>
struct overloaded : Ts... {
    using Ts::operator()...;
};
template <class... Ts>
overloaded(Ts...) -> overloaded<Ts...>;

// =========================================================================
// 1. AST Node Primitives (Sum Type Constituents)
// =========================================================================

// Numeric literal: 42, 3.1415
struct Number {
    double value;
};

// Named variable: "x", "y", "theta"
struct Variable {
    std::string name;
};

// Unary negation: -(expr)
struct Negate {
    std::shared_ptr<Expression> target;
};

// Binary addition: (lhs + rhs)
struct Add {
    std::shared_ptr<Expression> lhs;
    std::shared_ptr<Expression> rhs;
};

// Binary subtraction: (lhs - rhs)
struct Sub {
    std::shared_ptr<Expression> lhs;
    std::shared_ptr<Expression> rhs;
};

// Binary multiplication: (lhs * rhs)
struct Mul {
    std::shared_ptr<Expression> lhs;
    std::shared_ptr<Expression> rhs;
};

// Binary division: (lhs / rhs)
struct Div {
    std::shared_ptr<Expression> lhs;
    std::shared_ptr<Expression> rhs;
};

// Exponentiation: (base ^ exponent)
struct Power {
    std::shared_ptr<Expression> base;
    double exponent;
};

// =========================================================================
// 2. The Core Algebraic Sum Type
// =========================================================================

struct Expression {
    std::variant<
        Number,
        Variable,
        Negate,
        Add,
        Sub,
        Mul,
        Div,
        Power
    > node;

    // Constructors for seamless AST creation
    Expression(Number n) : node(n) {}
    Expression(Variable v) : node(std::move(v)) {}
    Expression(Negate neg) : node(std::move(neg)) {}
    Expression(Add add) : node(std::move(add)) {}
    Expression(Sub sub) : node(std::move(sub)) {}
    Expression(Mul mul) : node(std::move(mul)) {}
    Expression(Div div) : node(std::move(div)) {}
    Expression(Power pow) : node(std::move(pow)) {}
};

// Helper factory functions
[[nodiscard]] inline Expression num(double v) { return Expression{Number{v}}; }
[[nodiscard]] inline Expression var(std::string name) { return Expression{Variable{std::move(name)}}; }

[[nodiscard]] inline Expression operator+(Expression lhs, Expression rhs) {
    return Expression{Add{std::make_shared<Expression>(std::move(lhs)), std::make_shared<Expression>(std::move(rhs))}};
}

[[nodiscard]] inline Expression operator-(Expression lhs, Expression rhs) {
    return Expression{Sub{std::make_shared<Expression>(std::move(lhs)), std::make_shared<Expression>(std::move(rhs))}};
}

[[nodiscard]] inline Expression operator*(Expression lhs, Expression rhs) {
    return Expression{Mul{std::make_shared<Expression>(std::move(lhs)), std::make_shared<Expression>(std::move(rhs))}};
}

[[nodiscard]] inline Expression operator/(Expression lhs, Expression rhs) {
    return Expression{Div{std::make_shared<Expression>(std::move(lhs)), std::make_shared<Expression>(std::move(rhs))}};
}

[[nodiscard]] inline Expression operator-(Expression expr) {
    return Expression{Negate{std::make_shared<Expression>(std::move(expr))}};
}

[[nodiscard]] inline Expression pow(Expression base, double exp) {
    return Expression{Power{std::make_shared<Expression>(std::move(base)), exp}};
}

} // namespace sym

#endif // SYMBOLIC_ENGINE_HPP

Listing 12–2: Pretty Printer and Evaluator (Evaluator.hpp):

In [ ]:
#ifndef EVALUATOR_HPP
#define EVALUATOR_HPP

//#include "SymbolicEngine.hpp"
#include <unordered_map>

namespace sym {

// Pretty Print Expression to Standard Mathematical Notation
[[nodiscard]] inline std::string to_string(const Expression& expr) {
    return std::visit(overloaded{
        [](const Number& n) -> std::string {
            if (n.value == static_cast<long long>(n.value)) {
                return std::format("{}", static_cast<long long>(n.value));
            }
            return std::format("{:.2f}", n.value);
        },
        [](const Variable& v) -> std::string {
            return v.name;
        },
        [](const Negate& neg) -> std::string {
            return std::format("-({})", to_string(*neg.target));
        },
        [](const Add& add) -> std::string {
            return std::format("({} + {})", to_string(*add.lhs), to_string(*add.rhs));
        },
        [](const Sub& sub) -> std::string {
            return std::format("({} - {})", to_string(*sub.lhs), to_string(*sub.rhs));
        },
        [](const Mul& mul) -> std::string {
            return std::format("({} * {})", to_string(*mul.lhs), to_string(*mul.rhs));
        },
        [](const Div& div) -> std::string {
            return std::format("({} / {})", to_string(*div.lhs), to_string(*div.rhs));
        },
        [](const Power& p) -> std::string {
            return std::format("({} ^ {})", to_string(*p.base), p.exponent);
        }
    }, expr.node);
}

// Numerical Evaluator over Environment Dictionary
using Environment = std::unordered_map<std::string, double>;

[[nodiscard]] inline double evaluate(const Expression& expr, const Environment& env) {
    return std::visit(overloaded{
        [](const Number& n) -> double {
            return n.value;
        },
        [&env](const Variable& v) -> double {
            if (!env.contains(v.name)) {
                throw std::runtime_error(std::format("Unbound variable in environment: '{}'", v.name));
            }
            return env.at(v.name);
        },
        [&env](const Negate& neg) -> double {
            return -evaluate(*neg.target, env);
        },
        [&env](const Add& add) -> double {
            return evaluate(*add.lhs, env) + evaluate(*add.rhs, env);
        },
        [&env](const Sub& sub) -> double {
            return evaluate(*sub.lhs, env) - evaluate(*sub.rhs, env);
        },
        [&env](const Mul& mul) -> double {
            return evaluate(*mul.lhs, env) * evaluate(*mul.rhs, env);
        },
        [&env](const Div& div) -> double {
            const double denom = evaluate(*div.rhs, env);
            if (denom == 0.0) {
                throw std::runtime_error("Evaluation Exception: Math division by zero.");
            }
            return evaluate(*div.lhs, env) / denom;
        },
        [&env](const Power& p) -> double {
            return std::pow(evaluate(*p.base, env), p.exponent);
        }
    }, expr.node);
}

} // namespace sym

#endif // EVALUATOR_HPP

Listing 12–3: Type-Driven Algebraic Simplifier and Optimizer
(Simplifier.hpp):

In [ ]:
#ifndef SIMPLIFIER_HPP
#define SIMPLIFIER_HPP

//#include "SymbolicEngine.hpp"

namespace sym {

[[nodiscard]] inline Expression simplify(const Expression& expr) {
    return std::visit(overloaded{
        [](const Number& n) -> Expression {
            return num(n.value);
        },
        [](const Variable& v) -> Expression {
            return var(v.name);
        },
        [](const Negate& neg) -> Expression {
            Expression simplified_target = simplify(*neg.target);
            if (auto* n = std::get_if<Number>(&simplified_target.node)) {
                return num(-n->value);
            }
            return -simplified_target;
        },
        [](const Add& add) -> Expression {
            Expression l = simplify(*add.lhs);
            Expression r = simplify(*add.rhs);

            auto* num_l = std::get_if<Number>(&l.node);
            auto* num_r = std::get_if<Number>(&r.node);

            // Constant Folding: 2 + 3 -> 5
            if (num_l && num_r) return num(num_l->value + num_r->value);
            // Identity Rules: x + 0 -> x; 0 + x -> x
            if (num_l && num_l->value == 0.0) return r;
            if (num_r && num_r->value == 0.0) return l;

            return l + r;
        },
        [](const Sub& sub) -> Expression {
            Expression l = simplify(*sub.lhs);
            Expression r = simplify(*sub.rhs);

            auto* num_l = std::get_if<Number>(&l.node);
            auto* num_r = std::get_if<Number>(&r.node);

            // Constant Folding: 5 - 2 -> 3
            if (num_l && num_r) return num(num_l->value - num_r->value);
            // Identity Rule: x - 0 -> x
            if (num_r && num_r->value == 0.0) return l;

            return l - r;
        },
        [](const Mul& mul) -> Expression {
            Expression l = simplify(*mul.lhs);
            Expression r = simplify(*mul.rhs);

            auto* num_l = std::get_if<Number>(&l.node);
            auto* num_r = std::get_if<Number>(&r.node);

            // Constant Folding: 4 * 3 -> 12
            if (num_l && num_r) return num(num_l->value * num_r->value);
            // Zero Annihilation: x * 0 -> 0; 0 * x -> 0
            if ((num_l && num_l->value == 0.0) || (num_r && num_r->value == 0.0)) return num(0.0);
            // Multiplicative Identity: x * 1 -> x; 1 * x -> x
            if (num_l && num_l->value == 1.0) return r;
            if (num_r && num_r->value == 1.0) return l;

            return l * r;
        },
        [](const Div& div) -> Expression {
            Expression l = simplify(*div.lhs);
            Expression r = simplify(*div.rhs);

            auto* num_l = std::get_if<Number>(&l.node);
            auto* num_r = std::get_if<Number>(&r.node);

            if (num_r && num_r->value == 0.0) {
                throw std::runtime_error("Simplification Error: Static division by zero.");
            }
            if (num_l && num_r) return num(num_l->value / num_r->value);
            if (num_l && num_l->value == 0.0) return num(0.0);
            if (num_r && num_r->value == 1.0) return l;

            return l / r;
        },
        [](const Power& p) -> Expression {
            Expression b = simplify(*p.base);
            if (p.exponent == 0.0) return num(1.0);
            if (p.exponent == 1.0) return b;
            if (auto* num_b = std::get_if<Number>(&b.node)) {
                return num(std::pow(num_b->value, p.exponent));
            }
            return pow(b, p.exponent);
        }
    }, expr.node);
}

} // namespace sym

#endif // SIMPLIFIER_HPP

Listing 12–4: Symbolic Differentiator (Differentiator.hpp):

In [ ]:
#ifndef DIFFERENTIATOR_HPP
#define DIFFERENTIATOR_HPP

//#include "SymbolicEngine.hpp"
//#include "Simplifier.hpp"

namespace sym {

[[nodiscard]] inline Expression differentiate(const Expression& expr, const std::string& var_name) {
    return std::visit(overloaded{
        [](const Number&) -> Expression {
            return num(0.0); // d/dx (c) = 0
        },
        [&var_name](const Variable& v) -> Expression {
            return (v.name == var_name) ? num(1.0) : num(0.0); // d/dx (x) = 1, d/dx (y) = 0
        },
        [&var_name](const Negate& neg) -> Expression {
            return -differentiate(*neg.target, var_name);
        },
        [&var_name](const Add& add) -> Expression {
            // (u + v)' = u' + v'
            return differentiate(*add.lhs, var_name) + differentiate(*add.rhs, var_name);
        },
        [&var_name](const Sub& sub) -> Expression {
            // (u - v)' = u' - v'
            return differentiate(*sub.lhs, var_name) - differentiate(*sub.rhs, var_name);
        },
        [&var_name](const Mul& mul) -> Expression {
            // Product Rule: (u * v)' = u' * v + u * v'
            Expression u = *mul.lhs;
            Expression v = *mul.rhs;
            Expression u_prime = differentiate(u, var_name);
            Expression v_prime = differentiate(v, var_name);

            return (u_prime * v) + (u * v_prime);
        },
        [&var_name](const Div& div) -> Expression {
            // Quotient Rule: (u / v)' = (u' * v - u * v') / (v ^ 2)
            Expression u = *div.lhs;
            Expression v = *div.rhs;
            Expression u_prime = differentiate(u, var_name);
            Expression v_prime = differentiate(v, var_name);

            return ((u_prime * v) - (u * v_prime)) / pow(v, 2.0);
        },
        [&var_name](const Power& p) -> Expression {
            // Generalized Power Rule: (u ^ n)' = n * (u ^ (n - 1)) * u'
            Expression u = *p.base;
            Expression u_prime = differentiate(u, var_name);

            return num(p.exponent) * pow(u, p.exponent - 1.0) * u_prime;
        }
    }, expr.node);
}

} // namespace sym

#endif // DIFFERENTIATOR_HPP

Listing 12–5: Demonstration and Verification Driver (main.cpp):

In [ ]:
//#include "SymbolicEngine.hpp"
//#include "Evaluator.hpp"
//#include "Simplifier.hpp"
//#include "Differentiator.hpp"
#include <iostream>
#include <format>

int main() {
    std::cout << "=========================================================\n";
    std::cout << "Symbolic Algebraic Engine and Type-Driven Differentiation\n";
    std::cout << "=========================================================\n\n";

    using namespace sym;

    // Define symbolic variables: x and y
    auto x = var("x");
    auto y = var("y");

    // -------------------------------------------------------------
    // Example 1: f(x) = 3x^2 + 5x + 7
    // -------------------------------------------------------------
    Expression f = (num(3.0) * pow(x, 2.0)) + (num(5.0) * x) + num(7.0);

    std::cout << std::format("Original f(x)             : {}\n", to_string(f));

    // Numerical evaluation at x = 4 -> f(4) = 3(16) + 20 + 7 = 75
    Environment env{{"x", 4.0}};
    std::cout << std::format("Evaluate f(4)             : {}\n", evaluate(f, env));

    // Differentiate: f'(x) = 6x + 5
    Expression raw_derivative = differentiate(f, "x");
    std::cout << std::format("Raw Derivative f'(x)      : {}\n", to_string(raw_derivative));

    // Simplify derivative using algebraic reduction rules
    Expression clean_derivative = simplify(raw_derivative);
    std::cout << std::format("Simplified f'(x)          : {}\n", to_string(clean_derivative));

    // Evaluate f'(4) -> 6(4) + 5 = 29
    std::cout << std::format("Evaluate f'(4)            : {}\n\n", evaluate(clean_derivative, env));

    // -------------------------------------------------------------
    // Example 2: Product Rule Demonstration: g(x) = (x^3) * (x + 2)
    // g'(x) = 3x^2(x + 2) + x^3(1) = 4x^3 + 6x^2
    // -------------------------------------------------------------
    Expression g = pow(x, 3.0) * (x + num(2.0));
    std::cout << std::format("Original g(x)             : {}\n", to_string(g));
    Expression g_prime = simplify(differentiate(g, "x"));
    std::cout << std::format("Simplified g'(x)          : {}\n", to_string(g_prime));
    std::cout << std::format("Evaluate g'(2)            : {}\n\n", evaluate(g_prime, Environment{{"x", 2.0}}));

    return 0;
}

In [ ]:
main(); // Invocation of main program for Jupyter notebook running the xeus-cpp kernel

#### 12.5.6 Program Output

```
=========================================================
Symbolic Algebraic Engine and Type-Driven Differentiation
=========================================================

Original f(x)             : (((3 * (x ^ 2)) + (5 * x)) + 7)
Evaluate f(4)             : 75
Raw Derivative f'(x)      : ((((0 * (x ^ 2)) + (3 * (2 * (x ^ 1) * 1))) + ((0 * x) + (5 * 1))) + 0)
Simplified f'(x)          : ((6 * x) + 5)
Evaluate f'(4)            : 29

Original g(x)             : ((x ^ 3) * (x + 2))
Simplified g'(x)          : (((3 * (x ^ 2)) * (x + 2)) + (x ^ 3))
Evaluate g'(2)            : 56
```

Observe how `simplify()` dynamically collapses raw differentiation noise
(such as `0 * (x ^ 2)` and `5 * 1`) down to clean mathematical
expressions like `(6 * x) + 5` through the type-matching logic of
`std::visit`.

### 12.6 Summary

1.  **Types Impose Semantic Order on Raw Memory:** Without types, binary
    memory has no intrinsic meaning. Types establish value domains,
    valid operations, and physical alignment.
2.  **Product Types ($A \times B$):** Model conjunction ("AND"). An
    aggregation where all component types coexist simultaneously
    (`struct`, `tuple`).
3.  **Sum Types ($A + B$):** Model disjunction ("OR"). An exclusive
    tagged union holding exactly one valid alternative at a time
    (`std::variant`, `std::optional`).
4.  **C++20 Concepts:** Elevate templates from unconstrained text
    substitution to verified semantic contracts checked at compile time
    with readable compiler diagnostics.
5.  **Pattern Matching with `std::visit`:** Provides exhaustive,
    type-safe compile-time dispatch over Sum Types without `vtable`
    indirection overhead.

### 12.7 Programming Challenges

#### Exercise 12.1: Zero-Overhead Dimensional Analysis (Phantom Types and SI Units)

**Context and Theory:**

In 1999, NASA lost the \$125-million Mars Climate Orbiter because one
engineering team used imperial units (pound-force-seconds) while another
used metric units (newton-seconds). In standard procedural code, both
quantities are represented by raw `double` variables, allowing
accidental assignments without compiler warnings.

In Programming Language Theory, we prevent this using **Dimensional
Analysis via Type-Level Arithmetic**.

Every physical quantity in classical mechanics can be expressed as a
monomial of base SI dimensions: **Mass ($M$)**, **Length ($L$)**, and
**Time ($T$)**:

$$\text{Dimension}(Q) = M^m \cdot L^l \cdot T^t \quad \text{where } m, l, t \in \mathbb{Z}$$

- Length: $M^0 L^1 T^0$
- Time: $M^0 L^0 T^1$
- Velocity: $M^0 L^1 T^{-1}$ (Length / Time)
- Acceleration: $M^0 L^1 T^{-2}$ (Velocity / Time)
- Force: $M^1 L^1 T^{-2}$ ($\text{Mass} \times \text{Acceleration}$)

When multiplying two quantities, their dimensional exponents add
($m_1 + m_2$). When dividing, they subtract ($m_1 - m_2$). Addition and
subtraction are defined **only** when two quantities have identical
dimensional exponents ($m_1 = m_2, l_1 = l_2, t_1 = t_2$).

By parameterising a type with Non-Type Template Parameters (NTTPs)
representing these exponents, the compiler performs dimensional checking
at compile time with **zero runtime overhead**.

**Specification:**

Implement a compile-time dimensional analysis engine:

1.  Define a template struct `Quantity<int Mass, int Length, int Time>`
    wrapping a single `double value;`.
2.  Define type aliases: `Mass`, `Length`, `Time`, `Velocity`,
    `Acceleration`, and `Force`.
3.  Overload `operator+` and `operator-` to operate only on identical
    types.
4.  Overload `operator*` and `operator/` to compute new dimension types
    via compile-time exponent arithmetic.
5.  Provide user-defined literals (e.g., `5.0_m`, `2.0_s`, `10.0_kg`).

``` cpp
template <int M, int L, int T>
struct Quantity {
    double value;
    constexpr explicit Quantity(double v) noexcept : value(v) {}
};

template <int M, int L, int T>
[[nodiscard]] constexpr auto operator+(Quantity<M, L, T> a, Quantity<M, L, T> b) noexcept {
    return Quantity<M, L, T>(a.value + b.value);
}

template <int M1, int L1, int T1, int M2, int L2, int T2>
[[nodiscard]] constexpr auto operator*(Quantity<M1, L1, T1> a, Quantity<M2, L2, T2> b) noexcept {
    return Quantity<M1 + M2, L1 + L2, T1 + T2>(a.value * b.value);
}

template <int M1, int L1, int T1, int M2, int L2, int T2>
[[nodiscard]] constexpr auto operator/(Quantity<M1, L1, T1> a, Quantity<M2, L2, T2> b) noexcept {
    return Quantity<M1 - M2, L1 - L2, T1 - T2>(a.value / b.value);
}
```

**Verification Tasks:**

- **Task 1 (Physical Law Verification):** Define $m = 10.0\text{ kg}$
  and $a = 9.8\text{ m/s}^2$. Compute $F = m \times a$. Verify that
  `decltype(F)` is `Force` and its numerical value is $98.0$.
- **Task 2 (Dimensional Invalidation Proof):** Attempt to compile
  `Length l = 5.0_m; Time t = 2.0_s; auto x = l + t;`. Verify that
  compilation fails with an explicit type mismatch error.
- **Task 3 (Zero Runtime Overhead Inspection):** Inspect the assembly
  for $F = m \times a$ on Compiler Explorer (`godbolt.org`) under `-O2`.
  Verify that the operation compiles down to a single scalar
  multiplication instruction (`vmulsd`), with all template metadata
  erased.

#### Exercise 12.2: The Cardinality of Types: Making Illegal States Unrepresentable

**Context and Theory:**

A foundational principle of type-driven design is: **Make illegal states
unrepresentable**.

The **Cardinality** of a type, denoted $|T|$, is the total count of
distinct valid states an instance can inhabit:

- Product Type: $|A \times B| = |A| \times |B|$
- Sum Type: $|A + B| = |A| + |B|$

Consider a network socket modelled using standard procedural product
types:

``` cpp
struct SocketProduct {
    bool is_connected;   // 2 states
    bool has_error;       // 2 states
    int  error_code;      // ~4,000,000,000 states
    char host[64];        // 256^64 states
}; // Cardinality: 2 * 2 * 2^32 * 256^64 ≈ Astronomical!
```

This model permits contradictory states: an application could have
`is_connected = true`, `has_error = true`, and `error_code = 404`
simultaneously. To guard against these combinations, defensive
procedural code must litter every function with runtime validation
checks.

By switching to a **Sum Type** (`std::variant`), we collapse the
cardinality strictly to the union of valid states:

$$\begin{aligned} \text{SocketSum} &= \text{Disconnected} + \text{Connecting} + \text{Connected} + \text{Failed} \\
|\text{SocketSum}| &= 1 + 1 + |\text{ActiveConnection}| + |\text{SocketError}| \end{aligned}$$

Contradictory states become impossible to express.

**Specification:**

Model a network connection state machine using sum types:

1.  Define distinct state structs: `StateDisconnected`,
    `StateConnecting{std::string host}`,
    `StateConnected{std::uint32_t session_id}`, and
    `StateFailed{int error_code}`.
2.  Alias
    `using ConnectionState = std::variant<StateDisconnected, StateConnecting, StateConnected, StateFailed>;`.
3.  Implement a pure transition function:
    `ConnectionState transition(ConnectionState current, NetworkEvent event);`
4.  Guarantee that invalid transitions (e.g., receiving data while in
    `StateDisconnected`) return a `StateFailed` result.

``` cpp
#include <variant>
#include <string>

struct StateDisconnected {};
struct StateConnecting { std::string host; };
struct StateConnected { std::uint32_t session_id; };
struct StateFailed { int error_code; };

using ConnectionState = std::variant<StateDisconnected, StateConnecting, StateConnected, StateFailed>;

enum class NetworkEvent { ConnectRequested, HandshakeSuccess, Timeout, Disconnect };

[[nodiscard]] ConnectionState transition(const ConnectionState& state, NetworkEvent ev) {
    return std::visit([ev](const auto& s) -> ConnectionState {
        using T = std::decay_t<decltype(s)>;
        if constexpr (std::is_same_v<T, StateDisconnected>) {
            if (ev == NetworkEvent::ConnectRequested) return StateConnecting{"127.0.0.1"};
        } else if constexpr (std::is_same_v<T, StateConnecting>) {
            if (ev == NetworkEvent::HandshakeSuccess) return StateConnected{1001};
            if (ev == NetworkEvent::Timeout) return StateFailed{-1};
        }
        return StateFailed{999}; // Illegal transition
    }, state);
}
```

**Verification Tasks:**

- **Task 1 (Happy Path Transition):** Advance a connection from
  `StateDisconnected` $\to$ `ConnectRequested` $\to$ `HandshakeSuccess`.
  Verify that `std::holds_alternative<StateConnected>` is true and
  `session_id == 1001`.
- **Task 2 (Invalid Transition Trapping):** Pass
  `NetworkEvent::HandshakeSuccess` directly to a `StateDisconnected`
  instance. Verify that the system transitions to `StateFailed` with
  error code `999`.
- **Task 3 (Exhaustive Compile-Time Inspection):** Add a new state
  `StateReconnecting` to the variant. Ensure that failing to handle this
  state inside the transition function triggers a compile-time
  exhaustiveness error.

#### Exercise 12.3: Recursive AST Modelling: Building an Exhaustive JSON Parser AST

**Context and Theory:**

Real-world schemas often require recursive structural definitions. An
array in JSON can contain objects, which themselves contain arrays,
which in turn contain numbers or strings.

In type theory, a recursive type definition takes the form:

$$\text{JSON} ::= \text{Null} \mid \text{Bool} \mid \text{Number} \mid \text{String} \mid \text{List}[\text{JSON}] \mid \text{Map}[\text{String}, \text{JSON}]$$

In C++, a `std::variant` cannot hold itself directly by value because
that would require infinite memory:

$$\text{sizeof}(\text{Variant}) = \max(\text{sizeof}(T_i)) + \text{Padding} \implies \infty$$

To represent recursive sum types, we break the infinite size cycle using
**Indirection via Smart Pointers** or dynamic containers (`std::vector`,
`std::unordered_map`), which allocate memory on the heap and store a
fixed-size 8-byte pointer inside the variant.

**Specification:**

Construct an Abstract Syntax Tree (AST) for JSON:

1.  Define a recursive variant representing JSON elements.
2.  Implement pretty-printing serialization:
    `std::string to_json_string(const JsonValue& val)`.
3.  Implement a type-safe querying helper:
    `std::optional<T> get_value<T>(const JsonValue& val)`.
4.  Ensure the visitor handles all variant alternatives exhaustively
    without using `if-else` or dynamic casting.

``` cpp
#include <variant>
#include <string>
#include <vector>
#include <unordered_map>
#include <memory>

struct JsonValue;
using JsonNull   = std::monostate;
using JsonBool   = bool;
using JsonNumber = double;
using JsonString = std::string;
using JsonArray  = std::vector<JsonValue>;
using JsonObject = std::unordered_map<std::string, JsonValue>;

struct JsonValue {
    std::variant<JsonNull, JsonBool, JsonNumber, JsonString, JsonArray, JsonObject> data;
    template <typename T>
    JsonValue(T&& val) : data(std::forward<T>(val)) {}
};
```

**Verification Tasks:**

- **Task 1 (Nested AST Serialisation):** Construct an AST representing
  `{"score": 42.5, "tags": ["admin", "user"]}`. Pass it to
  `to_json_string()`. Verify that the output string matches standard
  JSON syntax.
- **Task 2 (Type-Safe Query Extraction):** Query the `"score"` key from
  the AST. Verify that `get_value<JsonNumber>` returns `42.5`, while
  requesting `get_value<JsonString>` on the same key returns
  `std::nullopt`.
- **Task 3 (Memory Leak Verification):** Construct a deeply nested JSON
  object (depth = 100). Destroy the object. Attach the memory profiler
  from Chapter 10 and verify that all heap nodes are cleanly
  deallocated.

#### Exercise 12.4: Formal Semantic Contracts via C++20 Concepts

**Context and Theory:**

Prior to C++20, templates were unconstrained. If a developer passed an
incompatible type to a template, the error was detected deep inside the
instantiation call stack, resulting in lengthy, unhelpful compiler
errors.

**C++20 Concepts** introduce formal compile-time predicates over types.

A concept is defined using a `requires` expression, which can validate:

1.  **Simple Requirements:** Whether an expression compiles ($a + b$).
2.  **Type Requirements:** Whether associated type aliases exist
    (`typename T::value_type`).
3.  **Compound Requirements:** Whether an expression compiles, does not
    throw exceptions (`noexcept`), and returns a type satisfying another
    concept:
    $$\{ \text{expr} \} \text{ noexcept} \to \text{std::convertible\_to}\langle\text{Target}\rangle;$$
4.  **Nested Requirements:** Additional constraints verified via
    `requires Concept<T>;`.

In abstract algebra, a **Ring** $(R, +, \cdot)$ is a set equipped with
two binary operations satisfying:

- $(R, +)$ is an abelian group (additive identity $0$, additive inverse
  $-a$).
- $(R, \cdot)$ is a monoid (multiplicative identity $1$, associative).
- Multiplication distributes over addition:
  $a \cdot (b + c) = (a \cdot b) + (a \cdot c)$.

**Specification:**

Define formal algebraic concepts in Modern C++20:

1.  Define a concept `Ring<T>` verifying that type `T` supports
    addition, subtraction, negation, and multiplication, and can be
    constructed from the literals `0` and `1`.
2.  Implement a generic matrix multiplication template
    `template <Ring T, std::size_t N> struct Matrix;`.
3.  Provide an explicit verification test proving that built-in integers
    and a custom `ModularArithmetic<N>` type satisfy `Ring`, while
    non-multiplicative types fail.

``` cpp
#include <concepts>

template <typename T>
concept Ring = requires(T a, T b) {
    { a + b } -> std::same_as<T>;
    { a - b } -> std::same_as<T>;
    { -a }    -> std::same_as<T>;
    { a * b } -> std::same_as<T>;
    { T(0) }; // Additive identity
    { T(1) }; // Multiplicative identity
};

template <Ring T, std::size_t N>
struct SquareMatrix {
    T data[N][N];
};
```

**Verification Tasks:**

- **Task 1 (Concept Satisfaction):** Assert via
  `static_assert(Ring<int>)` and `static_assert(Ring<double>)` that
  built-in numeric primitives satisfy the algebraic ring contract.
- **Task 2 (Modular Arithmetic Ring):** Implement a `ModularInt<17>`
  class. Assert that it satisfies `Ring<ModularInt<17>>` and can be used
  to instantiate `SquareMatrix<ModularInt<17>, 3>`.
- **Task 3 (Actionable Compiler Error Verification):** Attempt to
  instantiate `SquareMatrix<std::string, 3>`. Verify that the compiler
  fails immediately, pointing directly to the unsatisfied `Ring`
  constraint rather than failing deep inside matrix math logic.

#### Exercise 12.5: The Result / Expected Monad for Error Propagation

**Context and Theory:**

Software engineering uses three primary error-handling mechanisms:

1.  **Error Codes (C-style):** Returned via integers. They are easily
    ignored by callers, lack semantic payload data, and break function
    composition.
2.  **Exceptions (C++-style):** Separate the error path from the happy
    path, but incur significant stack-unwinding machinery and introduce
    invisible control-flow exits.
3.  **The Result / Expected Monad (Functional Style):** Represents
    operations that either succeed with value $T$ or fail with error
    $E$.

Mathematically, a Result type is a Sum Type:

$$\text{Result}\langle T, E\rangle = \text{Success}(T) + \text{Failure}(E)$$

In Modern C++ (C++23), this is standardised as `std::expected<T, E>`.

Monadic methods allow functions returning `Result` to be composed
linearly:

- `.map(F)`: Transforms $T \to U$ if successful; passes error $E$
  through otherwise.
- `.and_then(F)`: Chains computations that themselves return
  `Result<U, E>`, short-circuiting on the first error.

**Specification:**

Implement a minimal `Result<T, E>` monad:

- Wrap `std::variant<T, E>` where index 0 is success and index 1 is
  error.
- Implement `is_ok()` and `is_err()`.
- Implement `.value()` and `.error()` with exception guards.
- Implement monadic chaining: `.map()` and `.and_then()`.
- Write a pipeline that parses a numeric string, validates that it is
  within range, and computes its square root, propagating errors without
  throwing exceptions.

``` cpp
#include <variant>
#include <functional>
#include <stdexcept>

template <typename T, typename E>
class Result {
    std::variant<T, E> m_data;
public:
    Result(T val) : m_data(std::move(val)) {}
    static Result<T, E> err(E error) { Result r; r.m_data = std::move(error); return r; }

    [[nodiscard]] bool is_ok() const noexcept { return m_data.index() == 0; }
    [[nodiscard]] const T& value() const {
        if (!is_ok()) throw std::logic_error("Attempted to access value of an Err Result");
        return std::get<0>(m_data);
    }
    [[nodiscard]] const E& error() const { return std::get<1>(m_data); }

    template <typename F>
    auto and_then(F&& f) const -> decltype(f(value())) {
        if (is_ok()) return f(value());
        return decltype(f(value()))::err(error());
    }
private:
    Result() = default;
};
```

**Verification Tasks:**

- **Task 1 (Happy Path Composition):** Parse `"64"`, check that it is
  positive, and compute its square root. Verify that `result.is_ok()` is
  true and `result.value() == 8.0`.
- **Task 2 (Error Short-Circuiting):** Parse an invalid string `"abc"`.
  Verify that the pipeline short-circuits immediately, returning an
  error enum `ParseError`, and does not execute the subsequent square
  root stage.
- **Task 3 (Domain Validation Error):** Parse `"-16"`. Verify that
  parsing succeeds, but the domain validation stage catches the negative
  value and returns a `DomainError`.

#### Exercise 12.6: Automatic Differentiation via Dual Numbers

**Context and Theory:**

Computing mathematical derivatives is central to machine learning,
physics engines, and graphics. There are three traditional ways to
compute derivatives:

1.  **Numerical Differences:** $\frac{f(x + h) - f(x)}{h}$. Suffers from
    floating-point rounding errors and step-size truncation errors.
2.  **Symbolic Differentiation:** Constructing and manipulating an AST
    (as built in Section 5). Exact, but computationally expensive to
    evaluate at runtime.
3.  **Forward-Mode Automatic Differentiation:** Evaluates the function
    and its exact analytical derivative simultaneously in a single pass
    using **Dual Numbers**.

A **Dual Number** extends the real numbers by adjoining an element
$\epsilon$ where $\epsilon \ne 0$, but:

$$\epsilon^2 = 0$$

An arbitrary dual number is written $z = a + b\epsilon$.

Now, consider the Taylor series expansion of an analytic function $f$
around $x$:

$$f(x + b\epsilon) = f(x) + f'(x)b\epsilon + \frac{f''(x)}{2!}b^2\epsilon^2 + \dots$$

Because $\epsilon^2 = 0$, all higher-order terms vanish completely:

$$f(x + \epsilon) = f(x) + f'(x)\epsilon$$

Evaluating a function with $x + 1\epsilon$ computes **the function value
in the real part** and **the exact analytical derivative in the dual
component** in a single pass.

**Specification:**

Implement a `Dual<T>` number type:

- Member fields: `T real` and `T dual`.
- Overload arithmetic operators (`+`, `-`, `*`, `/`) based on dual
  number algebra:
  $$(a + b\epsilon)(c + d\epsilon) = (ac) + (ad + bc)\epsilon$$
  $$\frac{a + b\epsilon}{c + d\epsilon} = \frac{a}{c} + \left(\frac{bc - ad}{c^2}\right)\epsilon$$
- Implement dual overloads for transcendental functions (`sin`, `cos`,
  `exp`, `pow`).
- Write a test function that computes the exact derivative of
  $f(x) = 3x^3 + \sin(x)$ at $x = 2.0$.

``` cpp
#include <cmath>

template <typename T>
struct Dual {
    T real{0};
    T dual{0};

    constexpr Dual(T r, T d = T{0}) noexcept : real(r), dual(d) {}
};

template <typename T>
[[nodiscard]] constexpr Dual<T> operator*(Dual<T> a, Dual<T> b) noexcept {
    return Dual<T>{ a.real * b.real, (a.real * b.dual) + (a.dual * b.real) };
}

template <typename T>
[[nodiscard]] Dual<T> sin(Dual<T> x) noexcept {
    return Dual<T>{ std::sin(x.real), x.dual * std::cos(x.real) };
}
```

**Verification Tasks:**

- **Task 1 (Polynomial Derivative):** Evaluate $f(x) = 4x^2 + 5x + 3$ at
  $x = 3.0$ using `Dual<double>(3.0, 1.0)`. Verify that
  `result.real == 54.0` ($f(3)$) and `result.dual == 29.0`
  ($f'(3) = 8(3) + 5 = 29$).
- **Task 2 (Chain Rule Transcendental Verification):** Evaluate
  $g(x) = \sin(x^2)$ at $x = \sqrt{\pi}$. Verify that
  $g'(\sqrt{\pi}) = 2\sqrt{\pi}\cos(\pi) = -2\sqrt{\pi} \approx -3.544907$.
- **Task 3 (Quotient Rule Verification):** Compute the derivative of
  $h(x) = \frac{x^2}{x + 1}$ at $x = 2$. Verify that the computed
  derivative matches the analytical solution:
  $$h'(2) = \frac{2(2)(3) - (4)(1)}{3^2} = \frac{8}{9} \approx 0.888889$$

#### Exercise 12.7: Pure State Machines via Type Transitions

**Context and Theory:**

In classical object-oriented programming, state machines are often
implemented using the GoF State Pattern, storing a mutable pointer to a
polymorphic base state (`IState*`).

However, this design has drawbacks:

- Every state change requires dynamic memory allocation or shared
  pointer juggling.
- It does not enforce compile-time verification of valid transitions.
- Methods for events that are invalid in the current state must either
  throw runtime exceptions or fail silently.

In Modern C++, we can model state machines using **Pure Type
Transitions**. Each state is a distinct, concrete type. State
transitions are modelled as free functions that take the current state
type **by value** (consuming it) and return a new state type:

$$\text{Transition}: \text{State}_A \times \text{Event} \to \text{State}_B$$

If an event is illegal for a given state, no corresponding function
signature exists, and attempting to trigger that transition produces a
**compile-time error**.

**Specification:**

Design a compile-time validated TCP connection state machine:

1.  Define states: `Closed`, `SynSent`, `Established`, and `FinWait`.
2.  Implement transition functions:
    - `connect(Closed, IPAddress) -> SynSent`
    - `acknowledge(SynSent) -> Established`
    - `terminate(Established) -> FinWait`
3.  Verify that calling `acknowledge()` on a `Closed` socket is a
    compile-time syntax error.
4.  Model a dynamic runtime wrapper using `std::variant` that can accept
    runtime network packets while delegating to the compile-time type
    transitions.

``` cpp
struct IPAddress { const char* addr; };

struct Closed {};
struct SynSent { IPAddress target; };
struct Established { IPAddress target; std::uint32_t socket_fd; };
struct FinWait {};

[[nodiscard]] constexpr SynSent connect(Closed, IPAddress target) noexcept {
    return SynSent{target};
}

[[nodiscard]] constexpr Established acknowledge(SynSent s) noexcept {
    return Established{s.target, 42};
}

[[nodiscard]] constexpr FinWait terminate(Established) noexcept {
    return FinWait{};
}
```

**Verification Tasks:**

- **Task 1 (Linear Handshake Sequence):** Execute the handshake
  sequence: `Closed` $\to$ `connect` $\to$ `acknowledge` $\to$
  `terminate`. Verify that the final object type is `FinWait`.
- **Task 2 (Compile-Time Illegal Transition Invalidation):** Attempt to
  compile `acknowledge(Closed{})`. Verify that the compiler rejects the
  code because no matching function signature exists.
- **Task 3 (Zero Runtime Overhead):** Verify on Compiler Explorer that
  the entire sequence of transitions compiles down to moving values into
  registers, with zero heap allocations and zero virtual dispatch
  lookups.

#### Exercise 12.8: Untyped Lambda Calculus Evaluator

**Context and Theory:**

Introduced by Alonzo Church in the 1930s, the **Lambda Calculus** is the
foundational mathematical model of computation. It is Turing-complete,
proving that all computation can be reduced to three syntactic forms:

1.  **Variable:** $x$ (an identifier).
2.  **Abstraction (Function Definition):** $\lambda x. e$ (a function
    taking parameter $x$ with body $e$).
3.  **Application:** $(e_1 \ e_2)$ (calling function $e_1$ with argument
    $e_2$).

Computation occurs via **$\beta$-reduction**: substituting the argument
expression $e_2$ for every free occurrence of parameter $x$ in the
function body $e_1$:

$$(\lambda x. e_1) \ e_2 \implies e_1[x \mapsto e_2]$$

Using lambda calculus, we can encode data structures, booleans (**Church
Booleans**), and numbers (**Church Numerals**) purely using functions:

$$\begin{aligned}
\text{TRUE} &= \lambda x. \lambda y. x \\
\text{FALSE} &= \lambda x. \lambda y. y \\
\text{AND} &= \lambda p. \lambda q. p \ q \ p
\end{aligned}$$

**Specification:**

Construct an untyped Lambda Calculus evaluator in Modern C++:

1.  Define an AST using `std::variant`: `Var{string}`,
    `Abs{string param, shared_ptr<Expr> body}`, and
    `App{shared_ptr<Expr> func, shared_ptr<Expr> arg}`.
2.  Implement capture-avoiding substitution:
    `Expr substitute(const Expr& body, const string& var, const Expr& replacement)`.
3.  Implement small-step $\beta$-reduction:
    `std::optional<Expr> step(const Expr& expr)`.
4.  Implement a full evaluation loop that reduces expressions to normal
    form.
5.  Encode Church Booleans and evaluate the expression
    $\text{AND} \ \text{TRUE} \ \text{FALSE}$.

``` cpp
#include <variant>
#include <string>
#include <memory>

struct Expr;
struct Var { std::string name; };
struct Abs { std::string param; std::shared_ptr<Expr> body; };
struct App { std::shared_ptr<Expr> func; std::shared_ptr<Expr> arg; };

struct Expr {
    std::variant<Var, Abs, App> node;
    template <typename T> Expr(T v) : node(std::move(v)) {}
};
```

**Verification Tasks:**

- **Task 1 (Identity Application):** Construct $((\lambda x. x) \ y)$.
  Reduce the expression. Verify that it simplifies to the variable $y$.
- **Task 2 (Church Boolean Selection):** Evaluate the expression
  $(\text{TRUE} \ A \ B) \equiv ((\lambda x. \lambda y. x) \ A \ B)$.
  Verify that it reduces to $A$.
- **Task 3 (Logical AND Evaluation):** Construct and evaluate
  $(\text{AND} \ \text{TRUE} \ \text{FALSE})$. Verify that the
  expression reduces to the normal form for $\text{FALSE}$
  ($\lambda x. \lambda y. y$).

#### Exercise 12.9: Compile-Time Fixed-Point Arithmetic Type

**Context and Theory:**

Embedded digital signal processors (DSPs) and flight-control
microcontrollers often lack a hardware Floating-Point Unit (FPU). On
these platforms, floating-point arithmetic must be emulated in software,
which is hundreds of times slower than native integer math.

The systems solution is **Fixed-Point Arithmetic**.

A fixed-point number stores a real value scaled by a fixed power of two:

$$X_{\text{stored}} = \text{round}(X_{\text{real}} \times 2^F)$$

Where $F$ is the number of **Fractional Bits**, and $I$ is the number of
**Integer Bits**.

$$\text{Format } Q_{I.F}: \text{Total Bits } N = I + F$$

- **Addition / Subtraction:** Two numbers with the same fractional
  scaling can be added or subtracted directly using native integer
  instructions:
  $$Z_{\text{stored}} = X_{\text{stored}} + Y_{\text{stored}}$$
- **Multiplication:** Multiplies the underlying values, which scales the
  result by $2^{2F}$. The product must be shifted back down by $F$ bits:
  $$Z_{\text{stored}} = \frac{X_{\text{stored}} \times Y_{\text{stored}}}{2^F} = (X_{\text{stored}} \times Y_{\text{stored}}) \gg F$$

**Specification:**

Implement a `FixedPoint<int IntegerBits, int FractionalBits>` class:

1.  Store values using `int32_t` as the underlying representation.
2.  Provide `constexpr` constructors converting from `double` and `int`.
3.  Overload arithmetic operators (`+`, `-`, `*`, `/`).
4.  Ensure multiplication widens intermediate calculations to `int64_t`
    to prevent overflow before shifting.
5.  Provide explicit conversion back to `double`.

``` cpp
#include <cstdint>

template <int IntBits, int FracBits>
class FixedPoint {
    std::int32_t m_value{0};
    static constexpr std::int32_t SCALE = 1 << FracBits;
public:
    constexpr FixedPoint() noexcept = default;
    constexpr explicit FixedPoint(double v) noexcept 
        : m_value(static_cast<std::int32_t>(v * SCALE + (v >= 0 ? 0.5 : -0.5))) {}

    [[nodiscard]] constexpr double to_double() const noexcept {
        return static_cast<double>(m_value) / SCALE;
    }

    [[nodiscard]] constexpr FixedPoint operator+(FixedPoint rhs) const noexcept {
        FixedPoint res;
        res.m_value = m_value + rhs.m_value;
        return res;
    }

    [[nodiscard]] constexpr FixedPoint operator*(FixedPoint rhs) const noexcept {
        FixedPoint res;
        const auto wide_prod = static_cast<std::int64_t>(m_value) * rhs.m_value;
        res.m_value = static_cast<std::int32_t>(wide_prod >> FracBits);
        return res;
    }
};
```

**Verification Tasks:**

- **Task 1 (Compile-Time Arithmetic):** Declare
  `constexpr auto a = FixedPoint<16, 16>(3.25);` and
  `constexpr auto b = FixedPoint<16, 16>(2.5);`. Verify at compile time
  via `static_assert` that `(a + b).to_double() == 5.75`.
- **Task 2 (Multiplication Accuracy):** Compute `a * b`
  ($3.25 \times 2.5 = 8.125$). Verify that `(a * b).to_double()`
  evaluates to $8.125$ within rounding precision ($\epsilon < 0.001$).
- **Task 3 (Zero FPU Instruction Inspection):** Compile a function
  computing `FixedPoint` multiplication on Compiler Explorer with `-O2`.
  Verify that the emitted assembly contains **no floating-point
  instructions** (e.g., no `xmm` registers or `mulsd`), using only
  64-bit integer multiplication (`imul`) and arithmetic right shifts
  (`sar`).

#### Exercise 12.10: Heterogeneous Container Architectures: Type Erasure vs Sum Types

**Context and Theory:**

Systems often need to process collections of diverse geometric or
graphical shapes (e.g., `Circle`, `Rectangle`, `Triangle`) in a single
contiguous list.

Engineers typically choose between two architectural approaches:

1.  **Dynamic Polymorphism (Open Hierarchy):**
    `std::vector<std::unique_ptr<Shape>>` where each shape derives from
    an abstract base class with a virtual `area()` method. Each object
    is allocated independently on the dynamic heap.
2.  **Sum Types (Closed Variant):**
    `std::vector<std::variant<Circle, Rectangle, Triangle>>` where
    concrete shape value structs are stored **directly inline within the
    vector's contiguous memory buffer**. Operations are dispatched via
    `std::visit`.

<figure id="fig:memory-fragmentation-vs-variant-array">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NDYiIGhlaWdo
dD0iMTI4LjgiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNTQ2IDEyOC44Ij4KICA8c3R5
bGU+LnN2Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xlLCAuc3ZnYm9iIHJl
Y3QsIC5zdmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJva2Utd2lkdGg6IDEu
NTsKICBzdHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAgc3Ryb2tlLWxpbmVj
YXA6IHJvdW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdib2IgdGV4dCB7CiAg
d2hpdGUtc3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWlseTogQ291cmllciBO
ZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZTsKICBmb250LXNpemU6IDEzcHg7Cn0KCi5z
dmdib2IgcmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6IHRyYW5zcGFyZW50
Owp9Cgouc3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7Cn0KCi5zdmdib2Ig
LmZpbGxlZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxlZCB7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5ub2ZpbGwgewogIGZp
bGw6IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93IHsKICBtYXJrZXIt
ZW5kOiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fycm93IHsKICBtYXJr
ZXItc3RhcnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2RpYW1vbmQgewog
IG1hcmtlci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0
X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5lbmRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI29wZW5fY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6
IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2JpZ19vcGVuX2NpcmNs
ZSB7CiAgbWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFy
dF9tYXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYmlnX29wZW5f
Y2lyY2xlKTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9ImFycm93IiB2aWV3
Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJI
ZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9p
bnRzPSIwLDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtl
ciBpZD0iZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJr
ZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+
CiAgICAgIDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+PC9wb2x5Z29uPgog
ICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJl
Zlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0i
YXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNs
YXNzPSJmaWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJvcGVu
X2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAg
ICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIgdmlld0JveD0iMCAw
IDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIg
b3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIg
cj0iMyIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICA8L2RlZnM+
CiAgPHJlY3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9IjU0NiIgaGVpZ2h0
PSIxMjguOCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUiIHk9
IjI3LjMwMDAwMSIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMS45
NSIgeT0iNDIuOSIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMS45
NSIgeT0iNTguNSIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTcu
NTUwMDAxIiB5PSIyNy4zMDAwMDEiID4weDEwMjA8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iMTcuNTUwMDAxIiB5PSI0Mi45IiA+MHg0MDgwPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjE3LjU1MDAwMSIgeT0iNTguNSIgPjB4MjAxMDwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3Mi4xNSIgeT0iMjcuMzAwMDAxIiA+XTwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3Mi4xNSIgeT0iNDIuOSIgPl08
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzIuMTUiIHk9IjU4LjUiID5d
PC90ZXh0PgogIDxsaW5lIHgxPSI4NS44IiB5MT0iMjMuNDAwMDAyIiB4Mj0iMTA5LjIwMDAwNSIg
eTI9IjIzLjQwMDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIx
MDkuMjAwMDA1LDE5LjUgMTE3LDIzLjQwMDAwMiAxMDkuMjAwMDA1LDI3LjMwMDAwMSIgY2xhc3M9
ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSI4NS44IiB5MT0iMzkiIHgyPSIxMDkuMjAw
MDA1IiB5Mj0iMzkiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMTA5
LjIwMDAwNSwzNS4xMDAwMDIgMTE3LDM5IDEwOS4yMDAwMDUsNDIuOSIgY2xhc3M9ImZpbGxlZCI+
PC9wb2x5Z29uPgogIDxsaW5lIHgxPSI4NS44IiB5MT0iNTQuNjAwMDAyIiB4Mj0iMTA5LjIwMDAw
NSIgeTI9IjU0LjYwMDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRz
PSIxMDkuMjAwMDA1LDUwLjcgMTE3LDU0LjYwMDAwMiAxMDkuMjAwMDA1LDU4LjUiIGNsYXNzPSJm
aWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMjYuNzUi
IHk9IjI3LjMwMDAwMSIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MTI2Ljc1IiB5PSI0Mi45IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIxMjYuNzUiIHk9IjU4LjUiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjE0Mi4zNSIgeT0iMjcuMzAwMDAxIiA+Q2lyY2xlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjE5Ni45NTAwMSIgeT0iMjcuMzAwMDAxIiA+b248L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjIwLjM1IiB5PSIyNy4zMDAwMDEiID5IZWFwPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI1OS4zNSIgeT0iMjcuMzAwMDAx
IiA+XTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNDIuMzUiIHk9IjQy
LjkiID5SZWN0YW5nbGU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjIw
LjM1IiB5PSI0Mi45IiA+b248L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MjQzLjc1IiB5PSI0Mi45IiA+SGVhcDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIyODIuNzUiIHk9IjQyLjkiID5dPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjE0Mi4zNSIgeT0iNTguNSIgPlRyaWFuZ2xlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjIxMi41NSIgeT0iNTguNSIgPm9uPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjIzNS45NTAwMSIgeT0iNTguNSIgPkhlYXA8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjc0Ljk1IiB5PSI1OC41IiA+XTwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjk1IiB5PSIxMDUuMyIgPls8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTcuNTUwMDAxIiB5PSIxMDUuMyIgPkNpcmNs
ZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3Mi4xNSIgeT0iMTA1LjMi
ID5EYXRhPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjExMS4xNSIgeT0i
MTA1LjMiID5dWzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMzQuNTUi
IHk9IjEwNS4zIiA+UmVjdGFuZ2xlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjIxMi41NSIgeT0iMTA1LjMiID5EYXRhPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjI1MS41NSIgeT0iMTA1LjMiID5dWzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIyNzQuOTUiIHk9IjEwNS4zIiA+VHJpYW5nbGU8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzQ1LjE1IiB5PSIxMDUuMyIgPkRhdGE8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzg0LjE1MDAyIiB5PSIxMDUuMyIgPl08L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMS45NSIgeT0iMTEuNzAwMDAxIiA+
RHluYW1pYyBPT1A6IFZlY3RvciBvZiBQb2ludGVycyAoTWVtb3J5IEZyYWdtZW50YXRpb24gLyBD
YWNoZSBNaXNzZXMpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUi
IHk9Ijg5LjcwMDAwNSIgPlN1bSBUeXBlIFZhcmlhbnQ6IENvbnRpZ3VvdXMgSW5saW5lIEFycmF5
IChIaWdoIENhY2hlIExvY2FsaXR5KTwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 12–3: Memory Fragmentation versus Contiguous
Array</figcaption>
</figure>

The sum-type approach avoids heap allocations, improves spatial
locality, and enables compiler inlining during traversal.

**Specification:**

Design a comparative benchmark:

1.  Define shape types: `Circle{double r}`,
    `Rectangle{double w, double h}`, and `Triangle{double b, double h}`.
2.  Implement **Approach A**: An OOP hierarchy with an abstract base
    class `IShape` and a virtual `area()` method.
3.  Implement **Approach B**: Value structs stored inside a
    `std::variant`, processed using `std::visit`.
4.  Generate 1,000,000 random shapes in both containers.
5.  Benchmark the execution time required to compute the total
    aggregated area across both collections.

``` cpp
#include <variant>
#include <vector>

struct Circle { double r; };
struct Rectangle { double w, h; };
struct Triangle { double b, h; };

using ShapeVariant = std::variant<Circle, Rectangle, Triangle>;

[[nodiscard]] double compute_total_area(const std::vector<ShapeVariant>& shapes) noexcept {
    double total = 0.0;
    for (const auto& shape : shapes) {
        total += std::visit([](const auto& s) noexcept -> double {
            using T = std::decay_t<decltype(s)>;
            if constexpr (std::is_same_v<T, Circle>) return 3.14159265 * s.r * s.r;
            if constexpr (std::is_same_v<T, Rectangle>) return s.w * s.h;
            if constexpr (std::is_same_v<T, Triangle>) return 0.5 * s.b * s.h;
        }, shape);
    }
    return total;
}
```

**Verification Tasks:**

- **Task 1 (Area Calculation Parity):** Populate both collections with
  identical geometric values. Verify that both pipelines compute the
  exact same total area.
- **Task 2 (Memory Footprint Inspection):** Using the memory tracker
  from Chapter 10, verify that the `variant` vector creates **1
  allocation** (the contiguous buffer), whereas the OOP approach creates
  **1,000,001 allocations** (1 vector buffer + 1,000,000 individual heap
  nodes).
- **Task 3 (Traversal Speedup):** Benchmark area computation across the
  1,000,000 shapes. Verify that the `variant` approach is significantly
  faster than the OOP virtual pointer approach due to cache locality.

#### Exercise 12.11: Symbolic Calculus for Transcendental Expressions

**Context and Theory:**

In Section 5 of this chapter, we built a symbolic expression engine for
algebraic polynomials. To support scientific computing, an expression
engine must also handle **Transcendental Functions**
($\sin, \cos, \ln, \exp$).

Expanding the type system requires adding new nodes to the `Expression`
sum type and extending the differentiator with calculus rules:

- Derivative of Sine: $\frac{d}{dx}\sin(u) = \cos(u) \cdot u'$
- Derivative of Cosine: $\frac{d}{dx}\cos(u) = -\sin(u) \cdot u'$
- Derivative of Natural Logarithm: $\frac{d}{dx}\ln(u) = \frac{u'}{u}$
- Derivative of Exponential: $\frac{d}{dx}e^u = e^u \cdot u'$

Because our engine uses `std::variant`, extending the system requires
updating the visitor handlers inside `differentiate`, `simplify`, and
`evaluate`.

**Specification:**

Extend the Symbolic Engine from Section 5:

1.  Add AST node structs: `Sin`, `Cos`, `Ln`, and `Exp` (each holding a
    `std::shared_ptr<Expression> target`).
2.  Add these nodes to the `Expression` variant.
3.  Update `to_string()` to format functions in mathematical notation
    (e.g., `"sin(x)"`).
4.  Update `evaluate()` to compute numerical values using `<cmath>`.
5.  Update `differentiate()` to apply the calculus rules above.
6.  Verify the chain rule by computing the derivative of
    $f(x) = \ln(\cos(x))$.

``` cpp
// Additions to Symbolic Engine
struct Sin { std::shared_ptr<Expression> target; };
struct Cos { std::shared_ptr<Expression> target; };
struct Ln  { std::shared_ptr<Expression> target; };
struct Exp { std::shared_ptr<Expression> target; };

// Case addition for Differentiator
else if constexpr (std::is_same_v<T, Ln>) {
    // d/dx [ln(u)] = u' / u
    Expression u = *arg.target;
    Expression u_prime = differentiate(u, var_name);
    return u_prime / u;
}
```

**Verification Tasks:**

- **Task 1 (Transcendental Differentiation):** Compute the symbolic
  derivative of $f(x) = \sin(x)$. Simplify the result. Verify that
  `to_string()` yields `"cos(x)"`.
- **Task 2 (Chain Rule Differentiation):** Differentiate
  $g(x) = \exp(3x)$. Simplify the result. Verify that the derivative
  evaluates to $3e^{3x}$.
- **Task 3 (Numerical Calculus Consistency):** Evaluate the derivative
  of $h(x) = \ln(x)$ at $x = 2.0$. Verify that `evaluate()` returns
  $0.5$ ($\frac{1}{2}$).

#### Exercise 12.12: Dependent Type Simulation via Compile-Time Matrix Dimensions

**Context and Theory:**

In **Dependently Typed Languages** (such as Idris, Agda, or Coq), types
can depend on runtime values. A classic example is a matrix type
parameterised by its dimensions:

$$\text{Matrix}\langle\text{Rows}, \text{Cols}\rangle$$

In standard languages like C or Java, matrix dimensions are stored as
runtime fields. If an engineer attempts to multiply an $M \times K$
matrix by an $L \times N$ matrix where $K \ne L$, the error can only be
caught at runtime via an exception or assertion.

In Modern C++, we can simulate dependent typing using **Non-Type
Template Parameters (NTTPs)**:

$$\text{Matrix}\langle R_1, C_1\rangle \times \text{Matrix}\langle R_2, C_2\rangle \implies \text{Valid if and only if } C_1 == R_2$$

By constraining the multiplication operator so that it requires the
inner dimensions to match, dimension incompatibilities are caught **at
compile time**.

**Specification:**

Design a statically dimensioned matrix type:

1.  Define
    `template <typename T, std::size_t Rows, std::size_t Cols> class StaticMatrix;`.
2.  Store elements in a flat array `T m_data[Rows * Cols];`.
3.  Overload `operator+` requiring both operands to have identical
    `Rows` and `Cols`.
4.  Overload `operator*` taking `StaticMatrix<T, R1, C1>` and
    `StaticMatrix<T, R2, C2>`:
    - Require $C_1 == R_2$ using a compile-time constraint or
      `static_assert`.
    - Return a resulting matrix of dimension `StaticMatrix<T, R1, C2>`.

``` cpp
#include <cstddef>

template <typename T, std::size_t Rows, std::size_t Cols>
class StaticMatrix {
public:
    T data[Rows * Cols]{};

    template <std::size_t OtherCols>
    [[nodiscard]] constexpr auto operator*(const StaticMatrix<T, Cols, OtherCols>& rhs) const noexcept {
        StaticMatrix<T, Rows, OtherCols> result{};
        for (std::size_t r = 0; r < Rows; ++r) {
            for (std::size_t c = 0; c < OtherCols; ++c) {
                T sum = T{0};
                for (std::size_t k = 0; k < Cols; ++k) {
                    sum += data[r * Cols + k] * rhs.data[k * OtherCols + c];
                }
                result.data[r * OtherCols + c] = sum;
            }
        }
        return result;
    }
};
```

**Verification Tasks:**

- **Task 1 (Dimension Transformation Correctness):** Multiply a
  $2 \times 3$ matrix by a $3 \times 4$ matrix. Verify that `decltype`
  of the result is `StaticMatrix<T, 2, 4>`.
- **Task 2 (Compile-Time Mismatch Invalidation):** Attempt to compile:

``` cpp
StaticMatrix<double, 2, 3> A;
StaticMatrix<double, 4, 2> B;
auto C = A * B; // Inner dimensions 3 and 4 do not match!
```

Verify that the compiler rejects the code with a template deduction or
constraint failure. \* **Task 3 (Numerical Verification):** Multiply a
$2 \times 2$ matrix by its inverse. Verify that the product is a
$2 \times 2$ identity matrix ($I_2$).